In [48]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langchain_core.tools import tool

from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent

import base64

load_dotenv()

True

In [49]:
with open ("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [50]:
llm = ChatGroq(model = "qwen/qwen3.6-27b")

message = HumanMessage(content = [
    {"type": "image_url", "image_url":{"url":  f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report. Extract all test and print out the normal range with normal and name of the blood report with current number"}
])

response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from the provided blood test report image.
I need to identify:
1.  **Test Name**
2.  **Current Value**
3.  **Normal Range**

I will go through the report section by section.

**Section 1: COMPLETE BLOOD COUNT (CBC)**
*   **Hemoglobin:** 15.1 g/dL. Normal range: 13.5 - 17.5.
*   **Hematocrit:** 44%. Normal range: 41 - 53%.
*   **WBC:** 6.8 x 10^3/uL. Normal range: 4.5 - 11.0.
*   **Platelets:** 220 x 10^3/uL. Normal range: 150 - 400.

**Section 2: LIPID PANEL**
*   **Total Cholesterol:** 238 mg/dL. Normal range: < 200.
*   **LDL Cholesterol:** 162 mg/dL. Normal range: < 100.
*   **HDL Cholesterol:** 36 mg/dL. Normal range: > 40.
*   **Triglycerides:** 188 mg/dL. Normal range: < 150.

**Section 3: METABOLIC PANEL**
*   **Glucose (Fasting):** 92 mg/dL. Normal range: 70 - 99.
*   **HbA1c:** 5.3%. Normal range: < 5.7%.
*   **Creatinine:** 1.0 mg/dL. Normal range: 0.7 - 1.3.
*   **eGFR:** 82 mL/min. Normal range: > 60.

**Section 4: LIVER FUNCTION**

In [57]:
@tool
def get_diet_recomendation(condition: str) -> dict:
    """Given a health condition, return a diet plan. Condition must be one of these: normal, high_cholesterol, high_sugar. """
    diet_plans = {
        "high_cholesterol": {
            "eat": {"fruit", "vegetable", "whole grains", "lean protein"},
            "do not eat" : {"red meat", "full-fat dairy", "processed snaks"},
        },
        "high_sugar": {
            "eat": {"vegetable", "whole grains", "nuts", "lentil"},
            "do not eat" : {"white rice", "white sugar", "junk food"},
        },
        "Normal": {
            "eat": {"fruit", "vegetable", "whole grains", "lean protein"},
            "do not eat" : {"excessive sugar", "processed food", "trans fat"},
        },
    }
    return diet_plans.get(condition, diet_plans["Normal"])
    

agent = create_agent(
    llm,
    tools = [get_diet_recomendation],
    system_prompt = "you're very helpful nutrionist",
    checkpointer = InMemorySaver()
)

In [58]:
SYSTEM_PROMPT = """ 
You're helpful medical assitance and nutrition assistance.
From the blood work image. extract the number and the normal range, then categoriezed 
the condition as one of these: Normal, high_cholesterol, high_sugar
Then call the appropiate tool to retrive the and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools= [get_diet_recomendation],
    system_prompt = SYSTEM_PROMPT,
)

In [59]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text", "text": "Analyse this blood work report and suggest a diet plan"},
    ])]
})

print(result["messages"][-1].content)

Based on the blood work report provided for Rajesh Sharma, here is the analysis and the recommended diet plan.

### **Blood Work Analysis**

**Lipid Panel (High Cholesterol):**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) - **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) - **High**

**Metabolic Panel (Normal):**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - **Normal**
*   **HbA1c:** 5.3% (Normal: <5.7%) - **Normal**

**Conclusion:**
The patient has **High Cholesterol** (Hyperlipidemia), indicated by elevated total cholesterol, LDL, and triglycerides, along with low HDL. Blood sugar levels are healthy.

---

### **Recommended Diet Plan**

To help manage high cholesterol, here is the suggested diet plan:

**✅ Foods to Eat:**
*   **Whole grains:** Oats, brown rice, whole wheat bread.
*   **Fruit:** Berries, apples, citrus fruits.
*   **Vegetable:**